# Advanced Chunking Exercise

You will implement a RAG application for long and messy legal documents. You will implement the best practices you learned so far, including semantic chunking, and chunk enrichment. Then, you will implement semantic search and response generation with citation to the original documents.

In [1]:
!pip install semantic-router
!pip install semantic-chunkers
!pip install chromadb
!pip install pymupdf4llm
!pip install pymupdf
!pip install openai
!pip install rich
!pip install tqdm
!pip install sentence-transformers
!pip install transformers
!pip install torch

In [2]:
import sys

print(sys.executable)

C:\Users\2025\anaconda3\python.exe


In [3]:
!{sys.executable} -m pip install semantic-router

### Visual improvements

We will use [rich library](https://github.com/Textualize/rich) to make the output more readable, and supress warning messages.

In [4]:

from rich.console import Console

console = Console()

import warnings
warnings.filterwarnings("ignore")


In [5]:
import warnings

# Suppress warnings
warnings.filterwarnings('ignore')

## Loading complex PDF documents

You will load a complex legal PDF document from the [case.law](https://case.law/) website. This website has millions of legal documents, and we will load a random PDF file from that site with more than 1,000 pages. 

To parse the PDF file you will use a PDF processor library, [pymupdf4llm](https://pymupdf.readthedocs.io/en/latest/pymupdf4llm/), which makes it easy to extract text and other media from PDF files for RAG applications. 

In [6]:
import pymupdf4llm

import requests
import os

random_doc_number = 196
url = f"https://static.case.law/wash-app/{random_doc_number}.pdf"
response = requests.get(url)

data_folder = "data"
if not os.path.exists(data_folder):
    os.makedirs(data_folder)

with open(os.path.join(data_folder, f"{random_doc_number}.pdf"), "wb") as file:
    file.write(response.content)

md_text = pymupdf4llm.to_markdown(f"data/{random_doc_number}.pdf", page_chunks=True)

### Show a ramdom page from the document

Let's check a random page from the PDF document and print its image and the extracted text.

In [7]:
import fitz
from IPython.display import display, HTML

random_page_number = 149
## Convert the PDF to an PNG image
pdf_path = "data/196.pdf"
pdf_document = fitz.open(pdf_path)
page = pdf_document.load_page(random_page_number)  # Page numbering starts from 0
pix = page.get_pixmap()
pix.save("random_page.png")
pdf_document.close()

# Text content
text_content = f"""
<h3>Extracted Text</h3>
<p>{md_text[random_page_number]["text"]}</p>
"""

# HTML layout for two columns to show the image and text side by side
html_content = f"""
<div style="display: flex; align-items: center;">
    <div style="flex: 60%; padding: 5px;">
        <img src="{'random_page.png'}" style="max-width: 100%; height: auto;"/>
    </div>
    <div style="flex: 40%; padding: 5px;">
        {text_content}
    </div>
</div>
"""

# Display in Jupyter notebook
display(HTML(html_content))

You can see that the PDF processor extracts additional information on the document such as title, page count, etc. We can use this metadata for the metadata of our chunks in the vector database.

In [8]:
console.print(md_text[random_page_number])

defaultdict(<function make_page_chunk.<locals>.<lambda> at 0x0000023D1DD6F100>, {
    'metadata': {
        'format': 'PDF 1.3',
        'title': 'Washington Appellate Reports volume 196',
        'author': 'anonymous',
        'subject': 'unspecified',
        'keywords': '',
        'creator': 'Harvard Library Innovation Lab',
        'producer': 'ReportLab PDF Library - www.reportlab.com',
        'creationDate': "D:20190924090929+00'00'",
        'modDate': "D:20190924090929+00'00'",
        'trapped': '',
        'encryption': None,
        'file_path': 'data/196.pdf',
        'page_count': 1118,
        'page_number': 150
    },
    'toc_items': [],
    'page_boxes': [
        {'index': 0, 'class': 'page-header', 'bbox': (49, 41, 68, 50), 'pos': (0, 0)},
        {'index': 1, 'class': 'text', 'bbox': (48, 81, 351, 176), 'pos': (0, 0)},
        {'index': 2, 'class': 'text', 'bbox': (48, 182, 348, 328), 'pos': (0, 0)},
        {'index': 3, 'class': 'text', 'bbox': (48, 334, 350, 386), 'pos': (0, 0)},
        {'index': 4, 'class': 'text', 'bbox': (47, 391, 348, 543), 'pos': (0, 0)},
        {'index': 5, 'class': 'page-footer', 'bbox': (56, 562, 75, 569), 'pos': (0, 0)}
    ],
    'text': ''
})

## Split the documents into Chunks

You will use the statistical chunker that we used in the hands-on lab. However, we want an encoder that is trained on legal document and can generate better embedding vectors to improve the retrieval results. For this exercise you will an encoder from Hugging Face hub: https://huggingface.co/nlpaueb/legal-bert-base-uncased.

In [9]:
from semantic_router.encoders import HuggingFaceEncoder

encoder = HuggingFaceEncoder(
    ### YOUR CODE HERE ###-------------------------------------------------1
    name="nlpaueb/legal-bert-base-uncased"
)
console.print(encoder)


config.json:   0%|          | 0.00/1.02k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/222k [00:00<?, ?B/s]

2026-06-24 19:51:33 - huggingface_hub.utils._http - WARNING - _http.py:928 - _warn_on_warning_headers() - Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.


special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: nlpaueb/legal-bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


HuggingFaceEncoder(
    name='nlpaueb/legal-bert-base-uncased',
    score_threshold=0.5,
    type='huggingface',
    tokenizer_kwargs={},
    model_kwargs={},
    device='cpu'
)

In [10]:
from semantic_chunkers import StatisticalChunker
import logging

logging.disable(logging.CRITICAL)

chunker = StatisticalChunker(
    encoder=encoder,
    min_split_tokens=100,
    max_split_tokens=500,
)
console.print(chunker)

StatisticalChunker(
    name='statistical_chunker',
    encoder=HuggingFaceEncoder(
        name='nlpaueb/legal-bert-base-uncased',
        score_threshold=0.5,
        type='huggingface',
        tokenizer_kwargs={},
        model_kwargs={},
        device='cpu'
    ),
    splitter=RegexSplitter(
        regex_pattern='\n        # Negative lookbehind for word boundary, word char, dot, word char\n        
(?<!\\b\\w\\.\\w.)\n        # Negative lookbehind for single uppercase initials like "A."\n        
(?<!\\b[A-Z][a-z]\\.)\n        # Negative lookbehind for abbreviations like "U.S."\n        (?<!\\b[A-Z]\\.)\n     
# Negative lookbehind for abbreviations with uppercase letters and dots\n        (?<!\\b\\p{Lu}\\.\\p{Lu}.)\n      
# Negative lookbehind for numbers, to avoid splitting decimals\n        (?<!\\b\\p{N}\\.)\n        # Positive 
lookbehind for punctuation followed by whitespace\n        (?<=\\.|\\?|!|:|\\.\\.\\.)\\s+\n        # Positive 
lookahead for uppercase letter or opening quote at word boundary\n        (?="?(?=[A-Z])|"\\b)\n        # OR\n     
|\n        # Splits after punctuation that follows closing punctuation, followed by\n        # whitespace\n        
(?<=[\\"\\\'\\]\\)\\}][\\.!?])\\s+(?=[\\"\\\'\\(A-Z])\n        # OR\n        |\n        # Splits after punctuation 
if not preceded by a period\n        (?<=[^\\.][\\.!?])\\s+(?=[A-Z])\n        # OR\n        |\n        # Handles 
splitting after ellipses\n        (?<=\\.\\.\\.)\\s+(?=[A-Z])\n        # OR\n        |\n        # Matches and 
removes control characters and format characters\n        [\\p{Cc}\\p{Cf}]+\n        # OR\n        |\n        # 
Splits after punctuation marks followed by another punctuation mark\n        (?<=[\\.!?])(?=[\\.!?])\n        # 
OR\n        |\n        # Splits after exclamation or question marks followed by whitespace or end of string\n      
(?<=[!?])(?=\\s|$)\n    '
    ),
    threshold_adjustment=0.01,
    dynamic_threshold=True,
    window_size=5,
    plot_chunks=False,
    min_split_tokens=100,
    max_split_tokens=500,
    split_tokens_tolerance=10,
    enable_statistics=False,
    DEFAULT_THRESHOLD=0.5
)

### Chunking the full document text

We will concatenate the text from all the pages of the document. We will insert the page number between the pages to allow the retrieval and then then the generation steps to create direct citation to the relevant page in the long document.

In [11]:
# concatenated_text = " ".join([page["text"] + f"<page_break_{i}>" for i, page in enumerate(md_text)])

# #chunks =### YOUR CODE HERE ###-----------------------------------------2

# chunker([concatenated_text])

concatenated_text = " ".join(
    [page["text"] + f"<page_break_{i}>"
     for i, page in enumerate(md_text)]
)

chunks = chunker([concatenated_text])

print("DONE")

  0%|          | 0/5 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

[[Chunk(splits=['<page_break_0> <page_break_1> <page_break_2> <page_break_3> <page_break_4> <page_break_5> <page_break_6> <page_break_7> <page_break_8> <page_break_9> <page_break_10> <page_break_11> <page_break_12> <page_break_13> <page_break_14> <page_break_15> <page_break_16> <page_break_17> <page_break_18> <page_break_19> <page_break_20> <page_break_21> <page_break_22> <page_break_23> <page_break_24> <page_break_25> <page_break_26> <page_break_27> <page_break_28> <page_break_29> <page_break_30> <page_break_31> <page_break_32> <page_break_33> <page_break_34> <page_break_35> <page_break_36> <page_break_37> <page_break_38> <page_break_39> ##', '<page_break_40> <page_break_41> <page_break_42> <page_break_43> <page_break_44> <page_break_45> <page_break_46> ##', '<page_break_47> <page_break_48> <page_break_49> <page_break_50> <page_break_51> <page_break_52> <page_break_53> <page_break_54> <page_break_55> <page_break_56> <page_break_57> <page_break_58> ##', '<page_break_59> <page_break_60>

How many chunks were created?

In [19]:
### YOUR CODE HERE  ###-------------------------------------------------3
print(f"Number of chunks created: {len(chunks[0])}")

Number of chunks created: 29


Let's print a random chunk:

In [ ]:
console.print(chunks[0][5])

What is the average numebr of tokens in the chunks?

In [ ]:
### YOUR CODE HERE ###---------------------------------------------------------4
total_tokens = sum([chunk.token_count for chunk in chunks[0]])
average_tokens = total_tokens / len(chunks[0])
print(f"Average number of tokens: {average_tokens:.2f}")

## Enrich the chunk with context and metadata

We will iterate over all the chunks. This can take some time based on the number of chunks.

Since we want to be able to process a large number documents in our RAG system, we need to create a UUID that will used as the ID of the chunk within the vector database. The UUID is comprised of the URL of the document and the chunk index. This structure allows you to get a specific chunk index directly, whick will be improtant in the augmentation phase.

In [ ]:
import uuid
import re

doc_url = url
title = md_text[0]["metadata"]["title"]
# Enrich the metadata with filters that are relevant for future retrieval queries.
state = "Washington"

from tqdm import tqdm

def generate_uuid(doc_url, i):
    return str(uuid.uuid5(uuid.NAMESPACE_URL, f"{doc_url}/{i}"))

corpus_json = []
for i, chunk in tqdm(enumerate(chunks[0]), total=len(chunks[0]), desc="Processing chunks"):
    chunk_text = ' '.join(chunk.splits)
    # Extract the page number from the page breaks
    page_match = re.search(r'<page_break_(\d+)>', chunk_text)
    page = page_match.group(1) if page_match else 0
    chunk_uuid = generate_uuid(doc_url, i)
    corpus_json.append({
        "id": chunk_uuid,
        "document": chunk_text,
        # Add the title of the document to the chunk text for embedding
        "embedding": encoder([f"{title} \n {chunk_text}"])[0],
        "metadata" : {
            "title": title,
            "state": state,
            "doc_url": doc_url,
            "chunk_index": i,
            "page": page,
        }
    })


## Loading into a Vector Database

You will use a new vector data, [Chroma](https://github.com/chroma-core/chroma). It can illustrate the modularity of the RAG application, and the similar concepts across the providers.

### Creating the collection 

You will use the default values for this simpler exercise.

In [ ]:
import chromadb
# setup Chroma in-memory, for easy prototyping. Can add persistence easily!
client = chromadb.Client()

# Create collection. get_collection, create_collection, delete_collection also available!
collection_name = "legal-pdfs"
collection = client.get_or_create_collection(collection_name)


### Unserting the documents

We will use the embedding, metadata and documents that were calculated above.

In [ ]:
# collection.add(
#     documents=### YOUR CODE HERE ###,--------------------------------------------------5
#     embeddings=[obj["embedding"] for obj in corpus_json],
#     metadatas=### YOUR CODE HERE ###,
#     ### YOUR CODE HERE ###
# )
collection.add(
    documents=[obj["document"] for obj in corpus_json],
    embeddings=[obj["embedding"] for obj in corpus_json],
    metadatas=[obj["metadata"] for obj in corpus_json],
    ids=[obj["id"] for obj in corpus_json]
)

### Query the vector collection

We will add an example filter to the query based on the metadata that we created for each chunk (`{"state": "Washington"}`).

In [ ]:
query_text = "cases about loan default"
#query_embedding = ### YOUR CODE HERE ###-------------------------------------------------6
encoder([query_text])

In [ ]:
hits = collection.query(
    query_embeddings=query_embedding,
    n_results=5,
    where={"state": "Washington"},
)


## Augmentation Step

We suspect that the chunk context is too small and we want to concatenate the chunks around it, before we send the text to the generation step. 

In [ ]:
console.print(hits)

### Augmenting the search result

You will iterate over all the search results and prepare them to the generation step. The main augmentation is the concatenation of the sourounding chunks text.

In [ ]:
# define a variable to hold the search results with specific fields
search_results = []

# for document, metadata in zip(hits["documents"][0], hits["metadatas"][0]):------------------------7
#     doc_url = metadata["doc_url"]
#     chunk_index = metadata["chunk_index"]
#     doc_id = generate_uuid(doc_url, chunk_index)
#     # Calculate the chunk IDs of the previous and next chunks
#     previous_chunk_id = ### YOUR CODE HERE ###
#     next_chunk_id = ### YOUR CODE HERE ###
#     # Get the chunks from the vector collection with the chunk ids.
#     previous_chunk = collection.get(### YOUR CODE HERE ###)
#     next_chunk = ### YOUR CODE HERE ###
#     search_results.append({
#         # Concatenate the previous, current, and next document chunks to form a single document
#         "document": f"{previous_chunk['documents'][0]} {document} {next_chunk['documents'][0]}",
#         "metadata": metadata,
#     })
for document, metadata in zip(hits["documents"][0], hits["metadatas"][0]):
    doc_url = metadata["doc_url"]
    chunk_index = metadata["chunk_index"]
    doc_id = generate_uuid(doc_url, chunk_index)
    
    # حساب المعرفات واسترجاع الأجزاء المجاورة
    previous_chunk_id = generate_uuid(doc_url, chunk_index - 1) if chunk_index > 0 else doc_id
    next_chunk_id = generate_uuid(doc_url, chunk_index + 1)
    
    previous_chunk = collection.get(ids=[previous_chunk_id])
    next_chunk = collection.get(ids=[next_chunk_id])
    
    prev_text = previous_chunk['documents'][0] if previous_chunk['documents'] else ""
    next_text = next_chunk['documents'][0] if next_chunk['documents'] else ""
    
    search_results.append({
        "document": f"{prev_text} {document} {next_text}",
        "metadata": metadata,
    })

Let's print the first search result, before sending it to the generation model:

In [ ]:
console.print(search_results[0])

In [ ]:
from openai import OpenAI
from rich.panel import Panel
from rich.text import Text

client = OpenAI()
# system_message = """-----------------------------------------------------------------8
# You are a paralegal specialist. 
# ### YOUR CODE HERE ###
# ### YOUR CODE HERE ###
# """
system_message = """
You are a paralegal specialist. Your task is to analyze the retrieved legal contexts provided in the assistant message and answer the user's query precisely.
For every case or point you mention, you MUST provide a citation including the document title, volume, page number, and the source URL extracted from the metadata.
If the context does not contain enough information to answer, state that clearly.
"""
completion = client.chat.completions.create(
    model="gpt-4",
    messages=[
        {"role": "system", "content": system_message},
        {"role": "user", "content": query_text},
        {"role": "assistant", "content": str(search_results)}
    ]
)

response_text = Text(completion.choices[0].message.content)
styled_panel = Panel(
    response_text,
    title=f"Reply to '{query_text}'",
    expand=False,
    border_style="bright_yellow",
    padding=(1, 1)
)

console.print(styled_panel)

In [17]:
print("chunker exists:", "chunker" in globals())
print("concatenated_text exists:", "concatenated_text" in globals())
print("chunks exists:", "chunks" in globals())

chunker exists: True
concatenated_text exists: True
chunks exists: False


In [18]:
chunks = chunker([concatenated_text])

print("SUCCESS")

  0%|          | 0/5 [00:00<?, ?it/s]

SUCCESS
